# 05. Ridge Risk Scores and the Security Linear Program

Ridge regression produces a predicted property-crime rate \(\hat r_i\) on the frozen three mediators (Hoerl & Kennard, 1970). A linear program (a mathematical method for dividing a fixed budget subject to constraints) then allocates a fixed security budget across four CPTED control families (Bradley et al., 1977; Cozens et al., 2005).

The blocked place split and the coefficients \(\alpha_k\) are already frozen. We reuse `blocked_split.csv` and `alpha_k.csv`. The weight is ACS population.

Ridge follows the numbered D209 sequence with a different estimator: load, reuse the split, fit, evaluate against a mean-rate baseline, then write \(\hat r_i\). The program follows the D605 habit: name the variables, the constraints, and the objective before the solver runs.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pulp
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

HERE = Path.cwd()
if (HERE / "data" / "processed" / "place_year_clean.csv").exists():
    ROOT = HERE
elif (HERE.parent / "data" / "processed" / "place_year_clean.csv").exists():
    ROOT = HERE.parent
else:
    raise FileNotFoundError("Run this notebook from the Capstone root or from scripts/.")

PROCESSED = ROOT / "data" / "processed"
CLEAN = PROCESSED / "place_year_clean.csv"
FEATURE_LIST = PROCESSED / "feature_list.md"
SPLIT = PROCESSED / "blocked_split.csv"
ALPHA_K = PROCESSED / "alpha_k.csv"

DV = "property_crime_rate"
PREDICTORS = ["poverty_rate", "blau_heterogeneity", "pct_moved_year"]
BUDGET_USD = 10_000_000
RESIDUAL_CAP = 0.50
CONTROL_CEILING = 0.50
GROUP_CEILING = 0.50
HIGH_FLOOR = 0.25
PREVENT_FLOOR = 0.60
FORT_FLOOR = 0.20
SURV_FLOOR = 0.20
ALARM_FLOOR = 0.15
ALARM_CEILING = 0.25
PREVENT_IN_ELEVATED = 0.50
PREVENT_CONTROLS = ["fortification", "surveillance"]
TIERS = ["high", "mid", "low"]
ELEVATED = ["high", "mid"]

print("ROOT", ROOT)
print("budget USD", BUDGET_USD)


ROOT C:\Users\Owner\OneDrive\Capstone
budget USD 10000000


## File Gate

Stage 3 begins only when the cleaned table, the frozen feature list, the blocked split, and \(\alpha_k\) are on disk.

In [2]:
required = [CLEAN, FEATURE_LIST, SPLIT, ALPHA_K]
missing = [p for p in required if not p.exists()]
print("missing", missing)
assert not missing, missing

print("\nSTEP 1 LOAD FROZEN INPUTS")
print(FEATURE_LIST.read_text(encoding="utf-8"))

df = pd.read_csv(CLEAN)
split = pd.read_csv(SPLIT)
alpha_tbl = pd.read_csv(ALPHA_K)
print("clean rows", len(df), "split places", len(split), "alpha rows", len(alpha_tbl))
assert len(df) >= 200
assert set(PREDICTORS + [DV, "geoid", "acs_population"]).issubset(df.columns)
assert set(alpha_tbl["control_family"]) == {
    "fortification",
    "surveillance",
    "alarm",
    "authorization",
}
print("alpha_k (frozen; not refit)")
print(alpha_tbl[["control_family", "effectiveness", "median_unit_cost_usd", "alpha_k", "note"]].to_string(index=False))
ALPHAS = dict(zip(alpha_tbl["control_family"], alpha_tbl["alpha_k"]))
CONTROLS = ["fortification", "surveillance", "alarm", "authorization"]


missing []

STEP 1 LOAD FROZEN INPUTS
# Frozen feature list

Written by `scripts/03_bigset_qa_and_feature_freeze.ipynb`. This list does not change after notebook 03.

## Hypothesis predictors (locked)

- `poverty_rate` — ACS DP03, percent of all people below the poverty level
- `blau_heterogeneity` — Blau index from the DP05 groups locked in notebook 01 (Blau, 1977; Messner, 1986)
- `pct_moved_year` — ACS DP02, percent who lived in a different house one year earlier

## Dependent variable (locked)

- `property_crime_rate` — (burglary + larceny-theft + motor vehicle theft) / ACS population × 100,000
- Arson is a companion column and is not in the count (Federal Bureau of Investigation, 2019; notebook 02 coverage check).

## Official adds from the news pass

none

News themes that cleared five training-place stories mapped to fields already in the target or to LP families. No ACS vacancy column and no CBP establishment column were added. Headline counts are not features.

## Companion co

## Step 2. Reuse the Blocked Split

Places in New England and Middle Atlantic are the holdout (Liu et al., 2022; Varoquaux & Colliot, 2023). We merge the frozen place file and stop if any study place lacks a split label.

In [3]:
print("\nSTEP 2 REUSE BLOCKED SPLIT")
df = df.merge(split[["geoid", "census_division", "split"]], on="geoid", how="left")
n_unlabeled = int(df["split"].isna().sum())
print("unlabeled rows", n_unlabeled)
assert n_unlabeled == 0

train = df[df["split"] == "train"].copy()
holdout = df[df["split"] == "holdout"].copy()
print("train place-years", len(train), "places", train["geoid"].nunique())
print("holdout place-years", len(holdout), "places", holdout["geoid"].nunique())
print(df.groupby(["census_division", "split"]).size())
print("\nOgden")
print(
    df.loc[
        df["place"].str.contains("Ogden", case=False, na=False),
        ["year", "place", "state_abbr", "census_division", "split", DV],
    ].to_string(index=False)
)



STEP 2 REUSE BLOCKED SPLIT
unlabeled rows 0
train place-years 1939 places 684
holdout place-years 243 places 87
census_division     split  
East North Central  train      298
East South Central  train       86
Middle Atlantic     holdout    110
Mountain            train      222
New England         holdout    133
Pacific             train      627
South Atlantic      train      251
West North Central  train      176
West South Central  train      279
dtype: int64

Ogden
 year      place state_abbr census_division split  property_crime_rate
 2022 Ogden city         UT        Mountain train          2540.516864
 2023 Ogden city         UT        Mountain train          2089.154105
 2024 Ogden city         UT        Mountain train          1921.910929


The split places 1,939 place-years (684 places) in training and 243 place-years (87 places) in holdout. All three Northern Utah briefing cities — Ogden, Layton, and Logan — fall on the training side in the Mountain division. These are expanding cities (Layton +1.9%, Logan +3.1% over the panel), and their Ridge-predicted risk scores will feed the security-budget allocation in the exhibits notebook.

## Step 3. RidgeCV on the Training Blocks

Ridge keeps all three locked mediators and shrinks their coefficients (Hoerl & Kennard, 1970; James et al., 2013). Predictors are standardized on the training blocks only; the same scaler is applied to the holdout and to the full table. \(\lambda\) is chosen by `RidgeCV` on a log-spaced grid.

In [4]:
print("\nSTEP 3 RIDGECV ON TRAINING BLOCKS")
scaler = StandardScaler()
X_train = scaler.fit_transform(train[PREDICTORS])
X_hold = scaler.transform(holdout[PREDICTORS])
X_all = scaler.transform(df[PREDICTORS])
y_train = train[DV].to_numpy()
y_hold = holdout[DV].to_numpy()

ridge = RidgeCV(alphas=np.logspace(-2, 3, 20)).fit(X_train, y_train)
print(f"chosen lambda: {ridge.alpha_:.6f}")
print(f"training R-square: {ridge.score(X_train, y_train):.6f}")
print("coefficients on standardized predictors:")
for name, coef in zip(PREDICTORS, ridge.coef_):
    print(f"  {name}: {coef:.6f}")
print(f"intercept: {ridge.intercept_:.6f}")



STEP 3 RIDGECV ON TRAINING BLOCKS
chosen lambda: 26.366509
training R-square: 0.185041
coefficients on standardized predictors:
  poverty_rate: 422.970995
  blau_heterogeneity: 179.171535
  pct_moved_year: 110.874359
intercept: 2378.912134


RidgeCV selected λ = 26.37. On standardized predictors, poverty carries the largest coefficient (423), followed by heterogeneity (179) and mobility (111). Training R² is 0.185.

## Step 4. Holdout RMSE and MAE

The comparison is the training-block mean rate, predicted as a constant on the holdout (Varoquaux & Colliot, 2023). RMSE and MAE describe predictive error.

In [5]:
print("\nSTEP 4 HOLDOUT EVALUATION")
pred_hold = ridge.predict(X_hold)
base_hold = np.full_like(y_hold, fill_value=y_train.mean(), dtype=float)

def _metrics(y, yhat, label):
    rmse = float(np.sqrt(mean_squared_error(y, yhat)))
    mae = float(mean_absolute_error(y, yhat))
    print(f"{label}: RMSE {rmse:.4f}  MAE {mae:.4f}")
    return rmse, mae

ridge_rmse, ridge_mae = _metrics(y_hold, pred_hold, "Ridge")
base_rmse, base_mae = _metrics(y_hold, base_hold, "Mean-rate baseline")
print(f"holdout mean rate: {y_hold.mean():.4f}")
print(f"training mean rate: {y_train.mean():.4f}")
print(f"Ridge holdout mean prediction: {pred_hold.mean():.4f}")
print(f"Ridge RMSE minus baseline RMSE: {ridge_rmse - base_rmse:.4f}")



STEP 4 HOLDOUT EVALUATION
Ridge: RMSE 1050.1251  MAE 931.2221
Mean-rate baseline: RMSE 1020.6087  MAE 891.2517
holdout mean rate: 1850.7230
training mean rate: 2378.9121
Ridge holdout mean prediction: 2629.4755
Ridge RMSE minus baseline RMSE: 29.5165


The Northeast holdout has a lower mean property-crime rate than the training divisions. Ridge, fit on the higher-rate training blocks, overpredicts on New England and Middle Atlantic and records a larger RMSE and MAE than the training-mean baseline. That is the reason the split is geographic. The program uses the train-fit Ridge prediction on every study row, as locked (Hoerl & Kennard, 1970).

## Step 5. Predicted Rates for the Linear Program

We write \(\hat r_i\) on all 2,182 place-years. Negative predictions, if any, are clipped at zero because a rate cannot be negative. Risk tiers are a fixed cut of that score: **high** is the top decile, **mid** is the rest of the top quartile, **low** is the remaining 75 percent.

In [6]:
print("\nSTEP 5 SAVE RIDGE PREDICTIONS")
raw_hat = ridge.predict(X_all)
n_neg = int((raw_hat < 0).sum())
df["r_hat"] = np.maximum(raw_hat, 0.0)
print("negative predictions clipped", n_neg)
print(df["r_hat"].describe())

q90, q75 = np.quantile(df["r_hat"], [0.90, 0.75])
df["tier"] = np.where(
    df["r_hat"] >= q90, "high", np.where(df["r_hat"] >= q75, "mid", "low")
)
print(f"top-decile cut (q90): {q90:.4f}")
print(f"top-quartile cut (q75): {q75:.4f}")
print(df.groupby("tier").size().reindex(TIERS))

pred_path = PROCESSED / "ridge_predictions.csv"
pred_out = df[
    [
        "geoid",
        "year",
        "place",
        "state_abbr",
        "census_division",
        "split",
        "acs_population",
        DV,
        "r_hat",
        "tier",
    ]
].copy()
pred_out.to_csv(pred_path, index=False)
print("wrote", pred_path)
print("\nOgden predicted rates")
print(
    pred_out.loc[
        pred_out["place"].str.contains("Ogden", case=False, na=False)
    ].to_string(index=False)
)



STEP 5 SAVE RIDGE PREDICTIONS
negative predictions clipped 0
count    2182.000000
mean     2406.816309
std       514.006137
min      1292.502628
25%      2039.933289
50%      2348.270767
75%      2698.959756
max      4188.239647
Name: r_hat, dtype: float64
top-decile cut (q90): 3112.0259
top-quartile cut (q75): 2698.9598
tier
high     219
mid      327
low     1636
dtype: int64
wrote C:\Users\Owner\OneDrive\Capstone\data\processed\ridge_predictions.csv

Ogden predicted rates
  geoid  year      place state_abbr census_division split  acs_population  property_crime_rate       r_hat tier
4955980  2022 Ogden city         UT        Mountain train         86754.0          2540.516864 2320.980278  low
4955980  2023 Ogden city         UT        Mountain train         86973.0          2089.154105 2350.000560  low
4955980  2024 Ogden city         UT        Mountain train         87413.0          1921.910929 2389.086955  low


No negative predictions needed clipping. Predicted rates range from about 1,293 to 4,188. The top-decile cut is 3,112, producing 219 high-tier, 327 mid-tier, and 1,636 low-tier place-years.

Among the Northern Utah cities: **Ogden** (1,680–2,204 actual offenses) and **Layton** (1,131–1,410 offenses) fall in the **low tier** in all three years. **Logan** (634–786 offenses) falls in the **high tier** in all three years. Logan’s placement reflects its poverty and mobility levels, not its raw count. That contrast — low actual count, high predicted risk — is the finding that notebook 06 will unpack for the property owner.

## Linear Program (D605)

The rate is crimes per 100,000 residents. Exposure for group \(g\) is therefore the expected property-crime count on the place-year panel,

\[
E_g = \sum_{i \in g} \hat r_i \times w_i / 100{,}000,
\]

with \(w_i\) = ACS population (Lord & Nunes-Vaz, 2013, for the residual floor that follows). CBP establishments are not used.

**Planning budget.** \(B = \$10{,}000{,}000\). That is a fixed pot for the comparison.

### Variables

\(x_{g,k} \ge 0\) = dollars of control \(k\) assigned to risk group \(g\).

- \(g \in \{\text{high}, \text{mid}, \text{low}\}\)
- \(k \in \{\text{fortification}, \text{surveillance}, \text{alarm}, \text{authorization}\}\)

Twelve nonnegative variables. One program covers all tiers and cities.

### Objective

Minimize expected residual loss

\[
\sum_g \Big(E_g - \sum_k \alpha_k x_{g,k}\Big).
\]

\(\alpha_k\) is the frozen expected-loss share per dollar from notebook 03. What changes here is the feasible set: the solver must fund both a prevention layer and a notification layer.

### Control roles

Owners of plant, property, and equipment (PPE) buy a layered stack (Cozens et al., 2005; Pelco, n.d.).

- **Fortification** (delay / target hardening) and **surveillance** (watch) are preventive. Target hardening raises the effort of entry (Prenzler, 2009). CCTV, especially when it covers vehicle and lot space, is associated with lower property-crime counts in the 40-year evidence review (Piza et al., 2019).
- **Alarm** is reactive. In non-residential burglary it contributes to on-scene capture and later identification, not to stopping the first entry (Coupe & Kaur, 2005). Owners still need that notify-and-respond layer. They should not let it dominate the pot.
- **Authorization** (access control) stays in the variable list. It has no floor. The news freeze named alarm, fortification, and surveillance as the owner-facing families.

### Constraints

1. Total budget: \(\sum_{g,k} x_{g,k} \le B\).
2. Residual floor: \(E_g - \sum_k \alpha_k x_{g,k} \ge (1-m)E_g\) with \(m = 0.50\). Equivalently, \(\sum_k \alpha_k x_{g,k} \le 0.50\, E_g\). Layered controls are not fully additive (Lord & Nunes-Vaz, 2013).
3. Control ceiling: \(\sum_g x_{g,k} \le 0.50\, B\) so every family shares the budget.
4. Group ceiling: \(\sum_k x_{g,k} \le 0.50\, B\) so every tier shares the budget.
5. Highest-risk floor: \(\sum_k x_{\text{high},k} \ge 0.25\, B\).
6. Prevention floor: fortification plus surveillance \(\ge 0.60\, B\). Within that stack, fortification \(\ge 0.20\, B\) and surveillance \(\ge 0.20\, B\).
7. Reaction band: alarm \(\ge 0.15\, B\) and alarm \(\le 0.25\, B\).
8. Elevated-risk placement: at least \(0.50\, B\) of preventive dollars, and the full alarm floor, sit in the high and mid tiers. Watch and notify layers serve the higher tiers first.

### Baseline

Equal-share assigns \(B / 12\) to each \((g,k)\) cell. Residual loss under that mix is the planning comparison.

In [7]:
print("\nLP EXPOSURE")
df["exposure"] = df["r_hat"] * df["acs_population"] / 100000.0
E = df.groupby("tier")["exposure"].sum().reindex(TIERS)
print(E)
print("E total", float(E.sum()))


def residual_loss(mix: pd.Series, exposure: pd.Series, alphas: dict) -> float:
    red = 0.0
    for g in TIERS:
        for k in CONTROLS:
            red += alphas[k] * float(mix[(g, k)])
    return float(exposure.sum() - red)


def solve_lp(exposure: pd.Series, alphas: dict, budget: float):
    prob = pulp.LpProblem("security_budget", pulp.LpMinimize)
    x = {
        (g, k): pulp.LpVariable(f"x_{g}_{k}", lowBound=0)
        for g in TIERS
        for k in CONTROLS
    }
    prob += pulp.lpSum(
        exposure[g] - pulp.lpSum(alphas[k] * x[(g, k)] for k in CONTROLS)
        for g in TIERS
    )
    prob += pulp.lpSum(x.values()) <= budget, "budget"
    for g in TIERS:
        prob += (
            pulp.lpSum(alphas[k] * x[(g, k)] for k in CONTROLS)
            <= RESIDUAL_CAP * float(exposure[g]),
            f"residual_floor_{g}",
        )
        prob += (
            pulp.lpSum(x[(g, k)] for k in CONTROLS) <= GROUP_CEILING * budget,
            f"group_ceiling_{g}",
        )
    for k in CONTROLS:
        prob += (
            pulp.lpSum(x[(g, k)] for g in TIERS) <= CONTROL_CEILING * budget,
            f"control_ceiling_{k}",
        )
    prob += (
        pulp.lpSum(x[("high", k)] for k in CONTROLS) >= HIGH_FLOOR * budget,
        "high_floor",
    )
    prob += (
        pulp.lpSum(x[(g, k)] for g in TIERS for k in PREVENT_CONTROLS)
        >= PREVENT_FLOOR * budget,
        "prevent_floor",
    )
    prob += (
        pulp.lpSum(x[(g, "fortification")] for g in TIERS) >= FORT_FLOOR * budget,
        "fort_floor",
    )
    prob += (
        pulp.lpSum(x[(g, "surveillance")] for g in TIERS) >= SURV_FLOOR * budget,
        "surv_floor",
    )
    alarm_total = pulp.lpSum(x[(g, "alarm")] for g in TIERS)
    prob += alarm_total >= ALARM_FLOOR * budget, "alarm_floor"
    prob += alarm_total <= ALARM_CEILING * budget, "alarm_ceiling"
    prob += (
        pulp.lpSum(x[(g, k)] for g in ELEVATED for k in PREVENT_CONTROLS)
        >= PREVENT_IN_ELEVATED * budget,
        "prevent_in_elevated",
    )
    prob += (
        pulp.lpSum(x[(g, "alarm")] for g in ELEVATED) >= ALARM_FLOOR * budget,
        "alarm_in_elevated",
    )
    status = prob.solve(pulp.PULP_CBC_CMD(msg=False))
    status_name = pulp.LpStatus[status]
    mix = pd.Series({key: float(var.value()) for key, var in x.items()})
    return status_name, mix


status_name, solved_mix = solve_lp(E, ALPHAS, BUDGET_USD)
print("solver status", status_name)
assert status_name == "Optimal", status_name

mix_tbl = solved_mix.rename("dollars").reset_index()
mix_tbl.columns = ["tier", "control_family", "dollars"]
mix_wide = mix_tbl.pivot(index="tier", columns="control_family", values="dollars").reindex(
    TIERS
)[CONTROLS]
mix_wide["total"] = mix_wide.sum(axis=1)
print("\nSOLVED CONTROL MIX (dollars)")
print(mix_wide.round(2))
print("column totals")
print(mix_wide[CONTROLS].sum().round(2))
print("budget used", float(solved_mix.sum()))
prev_usd = float(mix_wide[PREVENT_CONTROLS].sum().sum())
alarm_usd = float(mix_wide["alarm"].sum())
print(f"preventive share: {prev_usd / BUDGET_USD:.2%}")
print(f"alarm share: {alarm_usd / BUDGET_USD:.2%}")

mix_path = PROCESSED / "control_mix.csv"
mix_tbl.to_csv(mix_path, index=False)
print("wrote", mix_path)



LP EXPOSURE
tier
high    1.182439e+06
mid     3.367544e+06
low     4.675129e+06
Name: exposure, dtype: float64
E total 9225112.445254184
solver status Optimal

SOLVED CONTROL MIX (dollars)
control_family  fortification  surveillance      alarm  authorization  \
tier                                                                    
high                      0.0     3500000.0  1500000.0            0.0   
mid                 5000000.0           0.0        0.0            0.0   
low                       0.0           0.0        0.0            0.0   

control_family      total  
tier                       
high            5000000.0  
mid             5000000.0  
low                   0.0  
column totals
control_family
fortification    5000000.0
surveillance     3500000.0
alarm            1500000.0
authorization          0.0
dtype: float64
budget used 10000000.0
preventive share: 85.00%
alarm share: 15.00%
wrote C:\Users\Owner\OneDrive\Capstone\data\processed\control_mix.csv


In [8]:
print("\nEQUAL-SHARE BASELINE")
cell = BUDGET_USD / (len(TIERS) * len(CONTROLS))
equal_mix = pd.Series(
    {(g, k): cell for g in TIERS for k in CONTROLS}
)
eq_tbl = equal_mix.rename("dollars").reset_index()
eq_tbl.columns = ["tier", "control_family", "dollars"]
eq_wide = eq_tbl.pivot(index="tier", columns="control_family", values="dollars").reindex(
    TIERS
)[CONTROLS]
eq_wide["total"] = eq_wide.sum(axis=1)
print(eq_wide.round(2))

solved_residual = residual_loss(solved_mix, E, ALPHAS)
equal_residual = residual_loss(equal_mix, E, ALPHAS)
print(f"\nsolved residual loss: {solved_residual:.4f}")
print(f"equal-share residual loss: {equal_residual:.4f}")
print(f"residual avoided vs equal-share: {equal_residual - solved_residual:.4f}")

eq_path = PROCESSED / "lp_equal_share.csv"
eq_tbl.to_csv(eq_path, index=False)
cmp_path = PROCESSED / "lp_comparison.csv"
pd.DataFrame(
    [
        {
            "budget_usd": BUDGET_USD,
            "exposure_total": float(E.sum()),
            "solved_residual": solved_residual,
            "equal_share_residual": equal_residual,
            "residual_avoided": equal_residual - solved_residual,
            "solver_status": status_name,
        }
    ]
).to_csv(cmp_path, index=False)
print("wrote", eq_path)
print("wrote", cmp_path)



EQUAL-SHARE BASELINE


control_family  fortification  surveillance      alarm  authorization  \
tier                                                                    
high                833333.33     833333.33  833333.33      833333.33   
mid                 833333.33     833333.33  833333.33      833333.33   
low                 833333.33     833333.33  833333.33      833333.33   

control_family       total  
tier                        
high            3333333.33  
mid             3333333.33  
low             3333333.33  

solved residual loss: 9207526.1627
equal-share residual loss: 9215673.3086
residual avoided vs equal-share: 8147.1459
wrote C:\Users\Owner\OneDrive\Capstone\data\processed\lp_equal_share.csv
wrote C:\Users\Owner\OneDrive\Capstone\data\processed\lp_comparison.csv


The solved mix is now a stack an owner can recognize. Prevention takes 85 percent of the pot: \$5 million of fortification in the mid tier and \$3.5 million of surveillance in the high tier. The notify layer takes the 15 percent alarm floor, also in the high tier, so the cities with the highest predicted rates get cameras and a call-out, alongside the delay layer. Authorization stays at zero. Equal-share still funds every cell, including the two weaker families, and leaves a larger residual (about 8,147 expected-count units). When surveillance \(\alpha_k\) is cut 25 percent, or alarm \(\alpha_k\) is raised 25 percent, alarm rises to its 25 percent cap and surveillance falls to 25 percent. The prevention-plus-notify shape holds.

## Sensitivity

Each frozen \(\alpha_k\) is rerun at ±25 percent, one family at a time. The other three coefficients stay at their frozen values. The budget and the constraint set do not change.

In [9]:
print("\nSENSITIVITY ON alpha_k")
rows = [
    {
        "scenario": "base",
        "family": "all",
        "factor": 1.0,
        "status": status_name,
        "residual": solved_residual,
        "residual_vs_equal": equal_residual - solved_residual,
        **{f"dollars_{k}": float(solved_mix.xs(k, level=1).sum()) for k in CONTROLS},
    }
]
for k in CONTROLS:
    for factor in (0.75, 1.25):
        alphas_s = dict(ALPHAS)
        alphas_s[k] = ALPHAS[k] * factor
        st, mix_s = solve_lp(E, alphas_s, BUDGET_USD)
        res_s = residual_loss(mix_s, E, alphas_s)
        res_eq = residual_loss(equal_mix, E, alphas_s)
        rows.append(
            {
                "scenario": f"{k}_x{factor}",
                "family": k,
                "factor": factor,
                "status": st,
                "residual": res_s,
                "residual_vs_equal": res_eq - res_s,
                **{f"dollars_{c}": float(mix_s.xs(c, level=1).sum()) for c in CONTROLS},
            }
        )
sens = pd.DataFrame(rows)
print(sens.to_string(index=False))
sens_path = PROCESSED / "lp_sensitivity.csv"
sens.to_csv(sens_path, index=False)
print("wrote", sens_path)



SENSITIVITY ON alpha_k


           scenario        family  factor  status     residual  residual_vs_equal  dollars_fortification  dollars_surveillance  dollars_alarm  dollars_authorization
               base           all    1.00 Optimal 9.207526e+06        8147.145884              5000000.0             3500000.0      1500000.0                    0.0
fortification_x0.75 fortification    0.75 Optimal 9.211693e+06        6063.812551              5000000.0             3500000.0      1500000.0                    0.0
fortification_x1.25 fortification    1.25 Optimal 9.203359e+06       10230.479217              5000000.0             3500000.0      1500000.0                    0.0
 surveillance_x0.75  surveillance    0.75 Optimal 9.207667e+06        8125.000000              5000000.0             2500000.0      2500000.0                    0.0
 surveillance_x1.25  surveillance    1.25 Optimal 9.207359e+06        8194.787618              5000000.0             3500000.0      1500000.0                    0.0
        al


wrote C:\Users\Owner\OneDrive\Capstone\data\processed\lp_sensitivity.csv


## Close

Ridge was fit on the training Census divisions and scored on New England and Middle Atlantic. Predicted rates were written for every study row. The linear program allocated the \(\$10\) million planning budget and was compared with equal-share. Sensitivity moved each \(\alpha_k\) by 25 percent.

Files: `data/processed/ridge_predictions.csv`, `data/processed/control_mix.csv`, `data/processed/lp_equal_share.csv`, `data/processed/lp_comparison.csv`, `data/processed/lp_sensitivity.csv`.

## References

Bradley, S. P., Hax, A. C., & Magnanti, T. L. (1977). *Applied mathematical programming*. Addison-Wesley.

Cozens, P. M., Saville, G., & Hillier, D. (2005). Crime prevention through environmental design (CPTED): A review and modern bibliography. *Property Management, 23*(5), 328–356. https://doi.org/10.1108/02637470510631483

Coupe, T., & Kaur, S. (2005). The role of alarms and CCTV in detecting non-residential burglary. *Security Journal, 18*(2), 29–42. https://doi.org/10.1057/palgrave.sj.8340198

Hoerl, A. E., & Kennard, R. W. (1970). Ridge regression: Biased estimation for nonorthogonal problems. *Technometrics, 12*(1), 55–67. https://doi.org/10.1080/00401706.1970.10488634

James, G., Witten, D., Hastie, T., & Tibshirani, R. (2013). *An introduction to statistical learning*. Springer. https://www.statlearning.com/

Liu, X., Kounadi, O., & Zurita-Milla, R. (2022). Incorporating spatial autocorrelation in machine learning models using spatial lag and eigenvector spatial filtering features. *ISPRS International Journal of Geo-Information, 11*(4), 242. https://doi.org/10.3390/ijgi11040242

Lord, S., & Nunes-Vaz, R. (2013). Designing and evaluating layered security. *International Journal of Risk Assessment and Management*.

Mitchell, S., Kean, A., Mason, A., Phillips, A., & Peschiera, F. (2009). *Optimization with PuLP*.

Pelco. (n.d.). *What is physical security? Policies, measures & examples*. https://www.pelco.com/blog/physical-security-guide

Piza, E. L., Welsh, B. C., Farrington, D. P., & Thomas, A. L. (2019). CCTV surveillance for crime prevention: A 40-year systematic review with meta-analysis. *Criminology & Public Policy, 18*(1), 135–159. https://doi.org/10.1111/1745-9133.12419

Prenzler, T. (2009). *Preventing burglary in commercial and institutional settings: A place management and partnerships approach*. Australian Research Council Centre of Excellence in Policing and Security.

Varoquaux, G., & Colliot, O. (2023). Evaluating machine learning models and their diagnostic value. In O. Colliot (Ed.), *Machine learning for brain disorders*. Humana. https://doi.org/10.1007/978-1-0716-3195-9_20
